# Lesson 4 - Wrapping the QA Agent into an A2A Server

In this lesson, you will take the `PolicyAgent` created in Lesson 3 and wrap it into an Agent2Agent (A2A) server. This makes the agent discoverable and callable by other agents using the A2A protocol. You will define the agent's identity using an `AgentCard`, describe its capabilities using `AgentSkill`, and set up the request handling logic.

## 4.1. Define the A2A Server

You will write the server code to a file named `a2a_policy_agent.py`. This script performs the following steps:
1.  **Imports**: Loads necessary modules including the A2A SDK (`a2a.server`).
2.  **Executor**: Defines `PolicyAgentExecutor`, which bridges the A2A request context to your `PolicyAgent` class.
3.  **Metadata**: Configures the `AgentSkill` (defining *what* it can do) and the `AgentCard` (defining *who* it is, including its URL).
4.  **Application**: Creates and runs an `A2AStarletteApplication` using `uvicorn`.


In [2]:
%%writefile ./a2a_policy_agent.py
from dotenv import load_dotenv
import os
import uvicorn

from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.apps import A2AStarletteApplication
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentSkill,
)
from a2a.utils import new_agent_text_message

from agents import PolicyAgent


class PolicyAgentExecutor(AgentExecutor):
    def __init__(self) -> None:
        self.agent = PolicyAgent()

    async def execute(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        prompt = context.get_user_input()
        response = self.agent.answer_query(prompt)
        message = new_agent_text_message(response)
        await event_queue.enqueue_event(message)

    async def cancel(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        pass

def main() -> None:
    print(f"Running A2A Health Insurance Policy Agent")
    _ = load_dotenv()
    PORT = int(os.environ.get("POLICY_AGENT_PORT", 9999))
    HOST = os.environ.get("AGENT_HOST", "localhost")

    skill = AgentSkill(
        id="insurance_coverage",
        name="Insurance coverage",
        description="Provides information about insurance coverage options and details.",
        tags=["insurance", "coverage"],
        examples=["What does my policy cover?", "Are mental health services included?"],
    )

    agent_card = AgentCard(
        name="InsurancePolicyCoverageAgent",
        description="Provides information about insurance policy coverage options and details.",
        url=f"http://{HOST}:{PORT}/",
        version="1.0.0",
        default_input_modes=["text"],
        default_output_modes=["text"],
        capabilities=AgentCapabilities(streaming=False),
        skills=[skill],
    )

    request_handler = DefaultRequestHandler(
        agent_executor=PolicyAgentExecutor(),
        task_store=InMemoryTaskStore(),
    )

    server = A2AStarletteApplication(
        agent_card=agent_card,
        http_handler=request_handler,
    )

    uvicorn.run(server.build(), host=HOST, port=PORT)

    
if __name__ == '__main__':
    main()
        

Overwriting ./a2a_policy_agent.py


## 4.2. Run the Policy A2A Server

Now to activate your configured A2A agent, run the agent server with `uv` from a local terminal:

- Run `uv sync` once to create the project environment.
- Run `uv run a2a_policy_agent.py` to start the server and activate your A2A agent.

**Note:** If you go to the next lesson, this agent will keep on running unless you manually stop it (`CTRL+C`) or the lab environment resets (on this learning platform, the lab environment resets after 30 minutes of inactivity or automatically every 2 hours).
If you stop the agent or the lab environment restarts, you don't need to come back to this lesson to launch the agent again. You will be provided with the needed files and terminal in each lesson, so you can always continue from where you left off.

In [3]:
from IPython.display import Markdown, display

display(Markdown("Run `uv sync` once, then start the server with `uv run a2a_policy_agent.py`."))

Run `uv sync` once, then start the server with `uv run a2a_policy_agent.py`.

In [5]:
%pip install uv

## 4.3. Resources

In [10]:
cd /content/drive/MyDrive/A2A-MCP

/content/drive/MyDrive/A2A-MCP


In [ ]:
!uv run a2a_policy_agent.py

Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
░░░░░░░░░░░░░░░░░░░░ [0/180] Installing wheels...                                        warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 180 packages in 6m 39s                            
Running A2A Health Insurance Policy Agent
INFO:     Started server process [70594]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:9999 (Press CTRL+C to quit)
INFO:     ::1:57540 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     ::1:57540 - "POST / HTTP/1.1" 200 OK
INFO:     ::1:36250 - "GET /.well-known/agent-card.json HTTP/1.1" 200 

- [Agent2Agent Protocol Documentation](https://a2a-protocol.org/)
- [A2A Python SDK Documentation](https://a2a-protocol.org/latest/sdk/python/api/)
- [Starlette Documentation](https://www.starlette.io/)
- [Equivalent notebook in the course repo](https://github.com/holtskinner/A2AWalkthrough/blob/main/2_A2AServer-PolicyAgent.ipynb)